# NHPP Baseball Hit Clustering - Real MLB Data

This notebook demonstrates the Hierarchical Bayesian NHPP model applied to **real MLB StatCast data**. All parameters are easily configurable for both small quick tests and large production runs.

**What's Different from Synthetic Example:**
- Uses actual MLB hit data from StatCast (2015-present)
- No "true" clusters to compare (we're discovering real patterns)
- Real temporal structure and hit patterns
- Scalable to full season or multiple seasons

**Setup Time**: 
- Data fetch + small run (100 hits, 100 MCMC): ~5-10 minutes
- Medium run (5000 hits, 1000 iterations): ~30-60 minutes  
- Large run (20000+ hits, 5000+ iterations): Plan for 2-4 hours

Adjust the parameters in the **Configuration** section to scale up/down as needed.

## 1. Setup Instructions

Make sure your conda environment is activated:
```bash
conda activate nhpp
jupyter notebook  # or: jupyter lab
```

**Note on data source:** This notebook fetches real data from Baseball Savant using the `pybaseball` library. First run may take longer (1-2 min) as data is downloaded.

In [1]:
# Import Required Libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.stats import multivariate_normal, invwishart, gamma
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add project to path
sys.path.insert(0, '/Users/mandujanoreyes/Documents/Research/NHPP')

# Import our modules
from src.config_templates import BASEBALL_CONFIG, build_priors
from src.data_extraction import quick_load_season, prepare_hits_spatiotemporal, assign_temporal_partition
from src.simulation.data_generation import initialize_chain_state
from src.core.config import Variables, Variables2, Cadena, SimPoisson, Tmax, L, alpha0
from src.core.bayesian_model import Post, AsignarZ_Todos

print("✓ All libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

✓ All libraries imported successfully
NumPy version: 2.2.6
Pandas version: 2.3.3


## 2. Configuration: Real Data Parameters

**Edit these parameters to control your analysis:**
- `MLB_SEASON`: Which season to analyze (2015-2026)
- `MIN_EXIT_VELOCITY`: Minimum exit velocity threshold (mph) - higher = harder hit balls
- `SAMPLE_SIZE`: If None, use all hits. Set to integer to subsample (useful for testing)
- `N_ITERATIONS`: MCMC chain length (100 = quick, 1000 = typical, 5000+ = thorough)

In [2]:
# ============================================================================
# REAL DATA CONFIGURATION - EDIT HERE
# ============================================================================

# Data source
MLB_SEASON = 2023  # Season to analyze [Options: 2015-2026]
MIN_EXIT_VELOCITY = 50  # Minimum exit velocity in mph [50 = all, 70+ = hard hits]
SAMPLE_SIZE = None  # None = use all data, or set to integer (e.g., 500, 5000)

# MCMC settings
N_ITERATIONS = 100  # Total chain length [small: 50, medium: 500, large: 2000+]
BURNIN = 20  # Iterations to discard
THIN = 1  # Save every nth iteration (1 = save all, 5 = save every 5th)

# Model configuration
config = BASEBALL_CONFIG.copy()
config['pT'] = 4  # Number of time partitions (quarter season)
config['ClustTiempo'] = 2  # Max clusters per time partition (conservative for real data)

# Derived settings
L = config['pT'] * config['ClustTiempo']

print("="*70)
print("REAL DATA CONFIGURATION")
print("="*70)
print(f"MLB Season:         {MLB_SEASON}")
print(f"Min Exit Velocity:  {MIN_EXIT_VELOCITY} mph")
print(f"Sample Size:        {SAMPLE_SIZE if SAMPLE_SIZE else 'All hits'}")
print(f"MCMC Iterations:    {N_ITERATIONS}")
print(f"Burn-in:            {BURNIN}")
print(f"Thinning:           {THIN}")
print(f"Temporal periods:   {config['pT']}")
print(f"Max clusters:       {L}")
print("="*70)

REAL DATA CONFIGURATION
MLB Season:         2023
Min Exit Velocity:  50 mph
Sample Size:        All hits
MCMC Iterations:    100
Burn-in:            20
Thinning:           1
Temporal periods:   4
Max clusters:       8


## 3. Fetch Real MLB Data from StatCast

This uses the `pybaseball` library to fetch actual hit data from Baseball Savant. First run may take 1-2 minutes.

In [ ]:
# Fetch real MLB data
print(f"\n{'='*70}")
print(f"Fetching MLB {MLB_SEASON} StatCast data...")
print(f"{'='*70}\n")

try:
    from pybaseball import statcast
    
    # Construct date range for MLB season (approximately March 28 - October 2)
    start_date = f"{MLB_SEASON}-03-28"
    end_date = f"{MLB_SEASON}-10-02"
    
    print(f"Date range: {start_date} to {end_date}")
    
    # Fetch the season using date range
    df = statcast(start_dt=start_date, end_dt=end_date)
    
    if df is None or df.empty:
        print(f"ERROR: No data available for season {MLB_SEASON}")
        print("Available seasons: 2015-2026")
        raise ValueError(f"Invalid season {MLB_SEASON}")
    
    print(f"✓ Downloaded {len(df):,} total pitches from {MLB_SEASON}")
    
    # Filter: balls in play only (have exit velocity and hit coordinates)
    df_inplay = df[df['launch_speed'].notna() & df['hc_x'].notna() & df['hc_y'].notna()].copy()
    print(f"✓ In-play hits: {len(df_inplay):,}")
    
    # Filter: exit velocity
    df_filtered = df_inplay[df_inplay['launch_speed'] >= MIN_EXIT_VELOCITY].copy()
    print(f"✓ Hits with exit velocity ≥ {MIN_EXIT_VELOCITY} mph: {len(df_filtered):,}")
    
    # Keep relevant columns - game_date should already be datetime from statcast
    hits = pd.DataFrame({
        'game_date': pd.to_datetime(df_filtered['game_date']),
        'hc_x': df_filtered['hc_x'],
        'hc_y': df_filtered['hc_y'],
        'exit_velocity': df_filtered['launch_speed'],
        'launch_angle': df_filtered['launch_angle'],
        'batter_id': df_filtered['batter'],
        'pitcher_id': df_filtered['pitcher'],
        'hit_type': df_filtered['type']
    })
    
    # Remove missing coordinates and dates
    hits = hits.dropna(subset=['hc_x', 'hc_y', 'game_date']).reset_index(drop=True)
    print(f"✓ Clean hits with coordinates: {len(hits):,}")
    
    # Sample if requested
    if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(hits):
        hits = hits.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
        print(f"✓ Sampled to {len(hits):,} hits for analysis")
    
    print(f"\nData Summary:")
    print(f"  Date range: {hits['game_date'].min().date()} to {hits['game_date'].max().date()}")
    print(f"  Exit velocity: {hits['exit_velocity'].min():.1f} - {hits['exit_velocity'].max():.1f} mph")
    print(f"  Batter count: {hits['batter_id'].nunique()}")
    print(f"  Pitcher count: {hits['pitcher_id'].nunique()}")
    
except ImportError:
    print("ERROR: pybaseball not installed!")
    print("Install with: pip install pybaseball")
    raise


Fetching MLB 2023 StatCast data...

Date range: 2023-03-28 to 2023-10-02
This is a large query, it may take a moment to complete


100%|██████████| 189/189 [01:48<00:00,  1.74it/s]


✓ Downloaded 723,106 total pitches from 2023


KeyError: 'inplay_run_expectancy'

## 4. Convert to Spatio-Temporal Format

Prepare the data for Bayesian inference.

In [8]:
# Convert to spatio-temporal format
min_date = hits['game_date'].min()
max_date = hits['game_date'].max()

# Prepare spatio-temporal point process
SimPoisson_true, Tmax = prepare_hits_spatiotemporal(hits, reference_date=min_date)

# Add temporal partitions
SimPoisson_true = assign_temporal_partition(SimPoisson_true, config['pT'], Tmax)

print(f"✓ Converted to spatio-temporal format")
print(f"  Total hits: {len(SimPoisson_true):,}")
print(f"  Field coordinates: x ∈ [{SimPoisson_true[:, 0].min():.1f}, {SimPoisson_true[:, 0].max():.1f}]")
print(f"                     y ∈ [{SimPoisson_true[:, 1].min():.1f}, {SimPoisson_true[:, 1].max():.1f}]")
print(f"  Season span: {Tmax:.1f} days")
print(f"  Temporal partitions: {config['pT']}")

NameError: name 'hits' is not defined

In [ ]:
# Visualize real data
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Hit locations (colored by exit velocity)
scatter = axes[0, 0].scatter(SimPoisson_true[:, 0], SimPoisson_true[:, 1], 
                             c=hits['exit_velocity'].values, cmap='YlOrRd', alpha=0.5, s=30)
axes[0, 0].set_xlabel('Field X (feet)', fontsize=11)
axes[0, 0].set_ylabel('Field Y (feet)', fontsize=11)
axes[0, 0].set_title(f'Hit Locations (colored by exit velocity)', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_aspect('equal')
cbar1 = plt.colorbar(scatter, ax=axes[0, 0])
cbar1.set_label('Exit Velocity (mph)', fontsize=10)

# Plot 2: Hit density heatmap
h2 = axes[0, 1].hist2d(SimPoisson_true[:, 0], SimPoisson_true[:, 1], bins=20, cmap='Blues')
axes[0, 1].set_xlabel('Field X (feet)', fontsize=11)
axes[0, 1].set_ylabel('Field Y (feet)', fontsize=11)
axes[0, 1].set_title('Hit Density Map', fontsize=12, fontweight='bold')
cbar2 = plt.colorbar(h2[3], ax=axes[0, 1])
cbar2.set_label('Count', fontsize=10)
axes[0, 1].set_aspect('equal')

# Plot 3: Time distribution
axes[1, 0].hist(SimPoisson_true[:, 2], bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[1, 0].set_xlabel('Time (days)', fontsize=11)
axes[1, 0].set_ylabel('Number of hits', fontsize=11)
axes[1, 0].set_title('Hit Distribution Across Season', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# Plot 4: Exit velocity distribution
axes[1, 1].hist(hits['exit_velocity'].values, bins=30, alpha=0.7, color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Exit Velocity (mph)', fontsize=11)
axes[1, 1].set_ylabel('Number of hits', fontsize=11)
axes[1, 1].set_title('Exit Velocity Distribution', fontsize=12, fontweight='bold')
axes[1, 1].axvline(MIN_EXIT_VELOCITY, color='red', linestyle='--', linewidth=2, label=f'Min threshold ({MIN_EXIT_VELOCITY} mph)')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis='y')

plt.suptitle(f'MLB {MLB_SEASON} Real Hit Data - {len(SimPoisson_true):,} hits', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f"✓ Data visualization complete")

## 5. Initialize Bayesian Model

Set up prior distributions and initialize the MCMC chain with reasonable starting values.

In [ ]:
# Build prior distributions
print("Building prior distributions...")
priors = build_priors(config)
print(f"✓ IW prior initialized: df={config['nu']}")
print(f"✓ Gaussian prior: μ={config['mu']}")
print(f"✓ Gamma prior: shape={config['pgamma']}, rate={config['k']}")

# Initialize MCMC state
print("\nInitializing MCMC chain state...")
SimPoisson_copy = SimPoisson_true[:, :3]  # Use first 3 columns [x, y, time]

init_state = initialize_chain_state(
    SimPoisson_copy, 
    config['pT'], 
    L, 
    config['alpha0'],
    config['xlim'], 
    config['Xlim'],
    config['ylim'], 
    config['Ylim'],
    priors['IW'], 
    priors['Gaus'], 
    priors['GammaPrior']
)

print(f"✓ Initial state initialized")
print(f"  Number of clusters initialized: {len(np.unique(init_state[3]))}")

# Configure global variables
Variables((
    Tmax, 
    config['pT'], 
    config['xlim'], 
    config['Xlim'],
    config['ylim'], 
    config['Ylim'], 
    SimPoisson_copy, 
    L,
    config['alpha0'], 
    priors['IW'], 
    priors['Gaus'],
    priors['GammaPrior']
))

# Initialize chain
chain = [init_state]
Variables2(chain)

print(f"✓ Model configuration complete. Ready for MCMC sampling.")

## 6. Run MCMC Sampling

Run the Bayesian model to estimate cluster assignments from real data.

In [ ]:
import time

# Run MCMC sampling
print(f"Starting MCMC sampling: {N_ITERATIONS} iterations on {len(SimPoisson_copy):,} hits...")
print("="*70)

start_time = time.time()
cluster_counts = []

for i in range(1, N_ITERATIONS):
    # Every 10% of iterations, print progress
    if i % max(1, N_ITERATIONS // 10) == 0:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        eta = (N_ITERATIONS - i) / rate if rate > 0 else 0
        print(f"Iteration {i:4d}/{N_ITERATIONS} | Elapsed: {elapsed:6.1f}s | ETA: {eta:6.1f}s | "
              f"Clusters: {len(np.unique(chain[-1][3]))}")
    
    # Simplified MCMC step: cluster reassignment
    try:
        # Assign each hit to a cluster
        Z_new = AsignarZ_Todos(i)
        
        # Create new state
        state_new = list(chain[-1])
        state_new[3] = Z_new  # Update cluster assignments
        
        chain.append(state_new)
        
        # Track cluster counts
        n_clusters = len(np.unique(Z_new))
        cluster_counts.append(n_clusters)
        
    except Exception as e:
        print(f"Warning at iteration {i}: {str(e)[:50]}...")
        # Continue with previous state
        chain.append(chain[-1])

elapsed = time.time() - start_time
print("="*70)
print(f"✓ MCMC sampling complete in {elapsed:.1f} seconds")
print(f"  Final cluster assignments: {len(np.unique(chain[-1][3]))} clusters")
print(f"  Chain length: {len(chain)}")

## 7. Analyze MCMC Results

Extract posterior cluster assignments and characterize the discovered hit patterns.

In [ ]:
# Extract posterior samples (after burn-in, with thinning)
burnin_idx = BURNIN
chain_posterior = chain[burnin_idx::THIN]

print(f"Posterior samples: {len(chain_posterior)} (after burn-in={BURNIN}, thin={THIN})")

# Get final cluster assignment and cluster frequencies
Z_posterior = chain_posterior[-1][3]
unique_clusters, counts = np.unique(Z_posterior, return_counts=True)

print(f"\nEstimated cluster structure (final iteration):")
print(f"{'Cluster':<10} {'Count':<10} {'Percentage':<12} {'Characteristics':<30}")
print("-" * 65)
for cluster_id, count in sorted(zip(unique_clusters, counts), key=lambda x: -x[1]):
    pct = 100 * count / len(Z_posterior)
    cluster_hits = hits.iloc[Z_posterior == cluster_id]
    
    mean_exit_vel = cluster_hits['exit_velocity'].mean()
    mean_launch_angle = cluster_hits['launch_angle'].mean()
    
    char = f"EV: {mean_exit_vel:.1f} mph, LA: {mean_launch_angle:.1f}°"
    
    print(f"{int(cluster_id):<10} {count:<10} {pct:>6.1f}%       {char:<30}")

# Compute trace statistics
cluster_evolution = [len(np.unique(state[3])) for state in chain_posterior]
print(f"\nNumber of clusters over posterior:")
print(f"  Mean: {np.mean(cluster_evolution):.2f}")
print(f"  Std: {np.std(cluster_evolution):.2f}")
print(f"  Range: [{np.min(cluster_evolution)}, {np.max(cluster_evolution)}]")

## 8. Visualize Discovered Hit Clusters

Compare posterior estimates and examine chain diagnostics.

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# Plot 1: Discovered clusters on field
ax1 = fig.add_subplot(gs[0, 0])
scatter1 = ax1.scatter(SimPoisson_true[:, 0], SimPoisson_true[:, 1], 
                       c=Z_posterior, cmap='tab10', alpha=0.6, s=40)
ax1.set_xlabel('Field X (feet)', fontsize=10)
ax1.set_ylabel('Field Y (feet)', fontsize=10)
ax1.set_title('Discovered Hit Clusters', fontweight='bold', fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_aspect('equal')
cbar1 = plt.colorbar(scatter1, ax=ax1)
cbar1.set_label('Cluster', fontsize=9)

# Plot 2: Clusters colored by exit velocity
ax2 = fig.add_subplot(gs[0, 1])
scatter2 = ax2.scatter(SimPoisson_true[:, 0], SimPoisson_true[:, 1], 
                       c=hits['exit_velocity'].values, cmap='YlOrRd', alpha=0.6, s=40)
ax2.set_xlabel('Field X (feet)', fontsize=10)
ax2.set_ylabel('Field Y (feet)', fontsize=10)
ax2.set_title('Colored by Exit Velocity', fontweight='bold', fontsize=11)
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')
cbar2 = plt.colorbar(scatter2, ax=ax2)
cbar2.set_label('EV (mph)', fontsize=9)

# Plot 3: Cluster size distribution
ax3 = fig.add_subplot(gs[0, 2])
est_sizes = np.unique(Z_posterior, return_counts=True)[1]
est_ids = np.unique(Z_posterior, return_counts=True)[0]
sorted_idx = np.argsort(-est_sizes)
ax3.barh(range(len(est_ids)), est_sizes[sorted_idx], color='steelblue', edgecolor='black', alpha=0.8)
ax3.set_xlabel('Number of hits', fontsize=10)
ax3.set_ylabel('Cluster', fontsize=10)
ax3.set_yticks(range(len(est_ids)))
ax3.set_yticklabels([f'C{int(est_ids[i])}' for i in sorted_idx])
ax3.set_title('Cluster Sizes', fontweight='bold', fontsize=11)
ax3.grid(alpha=0.3, axis='x')

# Plot 4: Trace of number of clusters
ax4 = fig.add_subplot(gs[1, :])
ax4.plot(cluster_evolution, linewidth=1.5, color='steelblue')
ax4.axvline(BURNIN, color='red', linestyle='--', linewidth=2, label=f'Burn-in (n={BURNIN})')
ax4.fill_between(range(BURNIN), 0, max(cluster_evolution)*1.1, alpha=0.2, color='red')
ax4.set_xlabel('MCMC Iteration', fontsize=11)
ax4.set_ylabel('Number of Clusters', fontsize=11)
ax4.set_title('Chain Trace: Number of Clusters', fontweight='bold', fontsize=12)
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)

# Plot 5: Spatial density of cluster assignments
ax5 = fig.add_subplot(gs[2, 0])
h5 = ax5.hist2d(SimPoisson_true[:, 0], SimPoisson_true[:, 1], bins=15, cmap='YlGnBu')
ax5.set_xlabel('Field X (feet)', fontsize=10)
ax5.set_ylabel('Field Y (feet)', fontsize=10)
ax5.set_title('Hit Density', fontweight='bold', fontsize=11)
cbar5 = plt.colorbar(h5[3], ax=ax5)
cbar5.set_label('Count', fontsize=9)
ax5.set_aspect('equal')

# Plot 6: Exit velocity by cluster
ax6 = fig.add_subplot(gs[2, 1])
cluster_exit_vels = [hits.iloc[Z_posterior == cid]['exit_velocity'].values for cid in np.unique(Z_posterior)]
bp = ax6.boxplot(cluster_exit_vels, labels=[f'C{int(cid)}' for cid in np.unique(Z_posterior)])
ax6.set_ylabel('Exit Velocity (mph)', fontsize=10)
ax6.set_xlabel('Cluster', fontsize=10)
ax6.set_title('Exit Velocity by Cluster', fontweight='bold', fontsize=11)
ax6.grid(alpha=0.3, axis='y')

# Plot 7: Launch angle by cluster
ax7 = fig.add_subplot(gs[2, 2])
cluster_launch_angles = [hits.iloc[Z_posterior == cid]['launch_angle'].values for cid in np.unique(Z_posterior)]
bp2 = ax7.boxplot(cluster_launch_angles, labels=[f'C{int(cid)}' for cid in np.unique(Z_posterior)])
ax7.set_ylabel('Launch Angle (°)', fontsize=10)
ax7.set_xlabel('Cluster', fontsize=10)
ax7.set_title('Launch Angle by Cluster', fontweight='bold', fontsize=11)
ax7.grid(alpha=0.3, axis='y')

plt.suptitle(f'MLB {MLB_SEASON} Analysis: {len(SimPoisson_true):,} hits, {N_ITERATIONS} MCMC iterations', 
             fontsize=14, fontweight='bold', y=0.995)
plt.show()

print("✓ Visualization complete")

## 9. Summary and Next Steps

### What We Did
1. **Fetched real MLB StatCast data** for the specified season
2. **Filtered and prepared data** by exit velocity and coordinate validity
3. **Initialized Bayesian NHPP model** with appropriate priors
4. **Ran MCMC sampling** to discover hit patterns
5. **Analyzed results** and visualized discovered clusters

### Discovered Hit Patterns

The model automatically discovered natural groupings of hits based on:
- **Spatial location**: Where on the field the hits land
- **Temporal dynamics**: How patterns evolve through the season
- **Physical characteristics**: Exit velocity and launch angle profiles

### Scaling to Production Runs

To run larger analyses with more iterations:

**Current test run:**
```python
N_ITERATIONS = 100
SAMPLE_SIZE = None  # All hits from season
```

**Medium analysis (30 min - 1 hour):**
```python
N_ITERATIONS = 500
MIN_EXIT_VELOCITY = 50
```

**Large production run (2-4 hours):**
```python
N_ITERATIONS = 2000
BURNIN = 500
THIN = 2
MIN_EXIT_VELOCITY = 60  # Focus on harder contact
```

### Save Your Results

```python
# Save the MCMC chain
np.save(f'results/chains/mlb_{MLB_SEASON}_chain.npy', chain)

# Save cluster assignments
np.save(f'results/summaries/mlb_{MLB_SEASON}_clusters.npy', Z_posterior)

# Save hit data
hits.to_csv(f'results/summaries/mlb_{MLB_SEASON}_hits.csv', index=False)
```

### Next Research Directions

1. **Compare seasons**: Analyze 2022-2024 to see how hit patterns evolve
2. **Pitcher-specific patterns**: Filter by pitcher hand (L/R) or team
3. **Batter stratification**: Analyze by batter skill level or handedness
4. **Uncertainty quantification**: Compute posterior credible intervals for cluster boundaries
5. **Optimal clustering**: Use Binder loss to find best point estimate clusters

In [ ]:
# Quick Command Reference

print("""
╔════════════════════════════════════════════════════════════════════╗
║                  QUICK REFERENCE FOR REAL DATA                    ║
╠════════════════════════════════════════════════════════════════════╣
║ JUPYTER NOTEBOOK                                                   ║
║   Start:       jupyter notebook                                   ║
║   This file:   notebooks/05_baseball_realdata.ipynb              ║
║                                                                    ║
║ ANALYZE DIFFERENT SEASONS                                          ║
║   Edit Section 2: MLB_SEASON = 2022  (or any 2015-2026)          ║
║   Re-run from "Fetch Real MLB Data" cell onward                  ║
║                                                                    ║
║ ADJUST FILTERS                                                     ║
║   min_exit_velocity = 60   # Higher = harder hits only            ║
║   SAMPLE_SIZE = 5000       # Subsample for quick testing          ║
║                                                                    ║
║ DATA QUALITY CHECKS                                                ║
║   Unique batters: hits['batter_id'].nunique()                     ║
║   Unique pitchers: hits['pitcher_id'].nunique()                   ║
║   Date range: hits['game_date'].min() to .max()                   ║
║   Exit velocity range: hits['exit_velocity'].describe()           ║
║                                                                    ║
║ SAVE RESULTS                                                       ║
║   np.save(f'results/chains/mlb_{MLB_SEASON}.npy', chain)         ║
║   hits.to_csv(f'results/summaries/mlb_{MLB_SEASON}_hits.csv')    ║
║                                                                    ║
║ DATA SOURCES                                                       ║
║   Website: https://baseballsavant.com/statcast                   ║
║   Python:  from pybaseball import statcast                       ║
║   Docs:    https://github.com/jldbc/pybaseball                   ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

# Summary statistics
print(f"\n✓ Run complete!")
print(f"  Season analyzed: {MLB_SEASON}")
print(f"  Total hits analyzed: {len(SimPoisson_true):,}")
print(f"  MCMC iterations: {N_ITERATIONS}")
print(f"  Posterior clusters discovered: {len(np.unique(Z_posterior))}")
print(f"  Processing time: {elapsed:.1f} seconds")
print(f"  Average time per iteration: {1000*elapsed/N_ITERATIONS:.0f} ms")